# Integração Numérica

**Métodos Numéricos — Engenharia Naval e Oceânica (Poli-USP)**

Neste notebook vamos estudar como calcular numericamente o valor de uma integral definida

$$I = \int_a^b f(x)\,dx$$

quando não é possível (ou não é conveniente) encontrar a primitiva de $f$ analiticamente, ou quando $f$ é conhecida apenas em pontos discretos (por exemplo, dados experimentais de um ensaio de tanque, uma curva de áreas seccionais de um navio, etc.).

Vamos construir, passo a passo e *do zero* (sem usar funções prontas de integração), os métodos:

1. Soma de Riemann (introdução ao conceito);
2. Regra dos Trapézios repetida;
3. Regra de Simpson repetida;

com visualizações geométricas e análise de convergência do erro.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configurações gerais de estilo dos gráficos
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Paleta de cores consistente usada em todo o notebook
COR_FUNCAO = 'black'
COR_RIEMANN = 'tab:blue'
COR_TRAPEZIO = 'tab:orange'
COR_SIMPSON = 'tab:green'
COR_EXATO = 'tab:red'

## 1. Motivação

Em muitas situações de engenharia não conseguimos (ou não queremos) calcular uma integral analiticamente:

- A função não possui primitiva elementar. Exemplo clássico: $\int e^{-x^2}dx$ (função erro), presente em estatística e em modelos de dispersão.
- A função só é conhecida em um conjunto discreto de pontos — por exemplo, a curva de áreas seccionais de um casco, obtida de um conjunto finito de balizas, ou uma série temporal de forças medidas em um ensaio.
- A expressão analítica existe, mas é complicada demais para ser integrada à mão.

A ideia central da **integração numérica** (também chamada de *quadratura numérica*) é aproximar a área sob a curva de $f(x)$ entre $a$ e $b$ por meio de formas geométricas simples, cuja área sabemos calcular exatamente:

- **Retângulos** → Soma de Riemann;
- **Trapézios** → Regra dos Trapézios;
- **Parábolas** → Regra de Simpson.

Quanto mais subdivisões (maior $n$, menor $h$) usarmos, mais a soma dessas áreas simples se aproxima da área real sob a curva — isto é, da integral. A pergunta de interesse em métodos numéricos não é só "o método converge?", mas **"com que velocidade ele converge conforme refinamos a malha?"** — é isso que vamos investigar a seguir.

## 2. Formulário

Abaixo estão reproduzidas as fórmulas do formulário oficial da disciplina para os dois métodos de integração numérica que serão estudados neste notebook. Aqui $h = \dfrac{b-a}{n}$ é o espaçamento entre os pontos (no caso de Simpson, $n$ denota o número de *pares* de subintervalos, totalizando $2n$ subintervalos).

### Fórmula dos Trapézios repetida $n$ vezes

$$\int_a^b f(x)\,dx \approx \frac{h}{2}\Big[f(x_0) + 2f(x_1) + \cdots + 2f(x_{n-1}) + f(x_n)\Big]$$

$$E_T = -\frac{nh^3}{12}f''(\sigma) = -\frac{(b-a)^3}{12n^2}f''(\sigma), \qquad \sigma \in [a,b]$$

### Fórmula de Simpson repetida $n$ vezes

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\Big[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 2f(x_{2n-2}) + 4f(x_{2n-1}) + f(x_{2n})\Big]$$

$$E_S = -\frac{nh^5}{90}f^{(IV)}(\sigma) = -\frac{(b-a)^5}{2880\,n^4}f^{(IV)}(\sigma), \qquad \sigma \in [a,b]$$

Observe que o erro teórico da Regra dos Trapézios é proporcional a $h^3$ por subintervalo (resultando em $O(h^2)$ no total, já que $n \sim 1/h$), enquanto o erro de Simpson é proporcional a $h^5$ por par de subintervalos (resultando em $O(h^4)$ no total). Essa diferença de ordem é o motivo pelo qual Simpson converge muito mais rápido — vamos comprovar isso numericamente mais adiante.

## 3. Soma de Riemann (ponto de partida)

Antes de chegar às fórmulas do formulário, vale relembrar a ideia mais simples de todas: a **soma de Riemann**. Dividimos o intervalo $[a,b]$ em $n$ subintervalos de largura $h=(b-a)/n$ e aproximamos a área sob a curva em cada subintervalo pela área de um retângulo de altura $f(x_i)$ (por exemplo, usando o ponto à esquerda de cada subintervalo):

$$\int_a^b f(x)\,dx \approx h\sum_{i=0}^{n-1} f(x_i)$$

Essa não é uma fórmula do formulário da disciplina, mas é o ponto de partida pedagógico: ela mostra visualmente a ideia de aproximar a área por formas simples, e por que métodos melhores (trapézios, parábolas) tendem a errar menos para o mesmo $n$. Vamos visualizar os retângulos para $f(x) = \sin(x)$ em $[0,\pi]$, cuja integral exata é $\int_0^\pi \sin(x)\,dx = 2$.

In [ ]:
def soma_riemann(f, a, b, n):
    """Calcula a soma de Riemann (ponto à esquerda) de f em [a, b] com n retângulos."""
    h = (b - a) / n
    xs = a + h * np.arange(n)  # pontos à esquerda de cada subintervalo: x0, x1, ..., x_{n-1}
    return h * np.sum(f(xs))


def plot_riemann(f, a, b, n, ax, titulo=''):
    """Desenha os retângulos da soma de Riemann sobre a curva de f."""
    x_fino = np.linspace(a, b, 400)
    ax.plot(x_fino, f(x_fino), color=COR_FUNCAO, linewidth=2, label='$f(x)$')

    h = (b - a) / n
    xs = a + h * np.arange(n)
    for xi in xs:
        ax.bar(xi, f(xi), width=h, align='edge', facecolor=COR_RIEMANN,
               edgecolor='k', alpha=0.4)

    aproximacao = soma_riemann(f, a, b, n)
    ax.set_title(f'{titulo}\n$n={n}$, soma $\\approx {aproximacao:.5f}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend()


f_teste = np.sin
a, b = 0.0, np.pi
exato = 2.0  # integral exata de sin(x) em [0, pi]

fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, n in zip(axs, [4, 12, 50]):
    plot_riemann(f_teste, a, b, n, ax, titulo='Soma de Riemann')

fig.suptitle('Convergência visual da soma de Riemann para $\\int_0^\\pi \\sin(x)\\,dx = 2$')
plt.tight_layout()
plt.show()

Note como, ao aumentar $n$ de 4 para 50, os retângulos "colam" cada vez mais na curva e a soma se aproxima do valor exato $2$. Essa é a ideia-mãe de toda a integração numérica: **refinar a malha reduz o erro**. A soma de Riemann, porém, converge de forma relativamente lenta (o erro cai proporcionalmente a $h$, ou seja, $O(h)$). Os próximos métodos usam formas geométricas melhores — trapézios e parábolas — para conseguir uma convergência muito mais rápida com o mesmo número de pontos.

## 4. Regra dos Trapézios repetida

### Dedução geométrica

Em vez de aproximar cada "fatia" da área por um retângulo, a Regra dos Trapézios liga os pontos $(x_i, f(x_i))$ e $(x_{i+1}, f(x_{i+1}))$ por um segmento de reta, formando um **trapézio**. A área de um trapézio de bases $f(x_i)$, $f(x_{i+1})$ e altura $h$ é

$$A_i = \frac{h}{2}\big[f(x_i) + f(x_{i+1})\big]$$

Somando a área de todos os $n$ trapézios ao longo de $[a,b]$, e notando que cada ponto interno $x_1, \ldots, x_{n-1}$ é compartilhado por dois trapézios vizinhos (por isso aparece multiplicado por 2), chegamos à fórmula do formulário:

$$\int_a^b f(x)\,dx \approx \frac{h}{2}\Big[f(x_0) + 2f(x_1) + \cdots + 2f(x_{n-1}) + f(x_n)\Big], \qquad h = \frac{b-a}{n}$$

Como o erro local em cada trapézio é $O(h^3)$ e há $n = O(1/h)$ trapézios, o erro total é $O(h^2)$ — ou seja, **dobrar o número de subdivisões reduz o erro por um fator de aproximadamente 4**.

Vamos implementar a fórmula do zero (sem usar `numpy.trapz`/`numpy.trapezoid`), visualizar os trapézios desenhados sob a curva, e depois verificar numericamente a ordem de convergência $O(h^2)$.

In [ ]:
def trapezios(f, a, b, n):
    """Regra dos Trapézios repetida, implementada do zero (sem numpy.trapz).

    Soma h/2 * [f(x0) + 2f(x1) + ... + 2f(x_{n-1}) + f(xn)]
    """
    h = (b - a) / n
    x = a + h * np.arange(n + 1)  # x0, x1, ..., xn
    y = f(x)
    soma = y[0] + y[-1] + 2.0 * np.sum(y[1:-1])
    return (h / 2.0) * soma


# Verificação rápida comparando com a função pronta do numpy
# (numpy >= 2.0 renomeou np.trapz para np.trapezoid; tentamos as duas por compatibilidade)
_funcao_trapz_numpy = getattr(np, 'trapezoid', None) or np.trapz

n_check = 1000
aprox_manual = trapezios(f_teste, a, b, n_check)
x_check = np.linspace(a, b, n_check + 1)
aprox_numpy = _funcao_trapz_numpy(f_teste(x_check), x_check)
print(f'Trapézios (implementação própria): {aprox_manual:.8f}')
print(f'Trapézios (numpy, verificação)    : {aprox_numpy:.8f}')
print(f'Valor exato                       : {exato:.8f}')

In [ ]:
def plot_trapezios(f, a, b, n, ax, titulo=''):
    """Desenha os trapézios usados na aproximação, sobrepostos à curva de f."""
    x_fino = np.linspace(a, b, 400)
    ax.plot(x_fino, f(x_fino), color=COR_FUNCAO, linewidth=2, label='$f(x)$')

    h = (b - a) / n
    x = a + h * np.arange(n + 1)
    y = f(x)
    for i in range(n):
        # cada trapézio é um polígono com vértices na base e no topo (reta ligando os dois pontos)
        xs_poligono = [x[i], x[i], x[i + 1], x[i + 1]]
        ys_poligono = [0, y[i], y[i + 1], 0]
        ax.fill(xs_poligono, ys_poligono, facecolor=COR_TRAPEZIO, edgecolor='k',
                alpha=0.4, linewidth=1)

    aproximacao = trapezios(f, a, b, n)
    ax.set_title(f'{titulo}\n$n={n}$, aproximação $\\approx {aproximacao:.5f}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend()


fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, n in zip(axs, [4, 8, 20]):
    plot_trapezios(f_teste, a, b, n, ax, titulo='Regra dos Trapézios')

fig.suptitle('Trapézios aproximando $\\int_0^\\pi \\sin(x)\\,dx = 2$')
plt.tight_layout()
plt.show()

### Análise de convergência do erro

Para confirmar experimentalmente que o erro da Regra dos Trapézios é $O(h^2)$, calculamos o erro absoluto $|I_{\text{aprox}} - I_{\text{exato}}|$ para vários valores de $n$ e plotamos $\log(\text{erro})$ contra $\log(n)$ (equivalentemente, contra $\log(h)$). Em escala log-log, uma relação de erro $\propto n^{-p}$ aparece como uma **reta de inclinação $-p$**. Esperamos uma inclinação próxima de $-2$.

In [ ]:
ns = np.array([2, 4, 8, 16, 32, 64, 128, 256, 512, 1024])
erros_trapezio = np.array([abs(trapezios(f_teste, a, b, n) - exato) for n in ns])

# Inclinação estimada por regressão linear em escala log-log: log(erro) = -p*log(n) + c
p_trapezio, c_trapezio = np.polyfit(np.log(ns), np.log(erros_trapezio), 1)
print(f'Ordem de convergência estimada (Trapézios): p ~= {-p_trapezio:.3f}  (teórico: 2)')

fig, ax = plt.subplots()
ax.loglog(ns, erros_trapezio, 'o-', color=COR_TRAPEZIO, label='Erro numérico')

# reta de referência com inclinação teórica -2, ajustada para passar perto dos dados
referencia = erros_trapezio[0] * (ns / ns[0])**(-2.0)
ax.loglog(ns, referencia, '--', color='gray', label='Referência $O(n^{-2})$')

ax.set_xlabel('$n$ (número de subintervalos)')
ax.set_ylabel('Erro absoluto')
ax.set_title(f'Convergência da Regra dos Trapézios (inclinação medida $\\approx {-p_trapezio:.2f}$)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Regra de Simpson repetida

### Dedução geométrica

A Regra de Simpson vai um passo além dos trapézios: em vez de ligar dois pontos consecutivos por uma reta, ela ajusta uma **parábola** (polinômio de grau 2) passando por **três** pontos consecutivos $(x_{2i}, f(x_{2i}))$, $(x_{2i+1}, f(x_{2i+1}))$, $(x_{2i+2}, f(x_{2i+2}))$, e integra essa parábola exatamente nesse par de subintervalos. Como uma parábola acompanha a curvatura da função muito melhor do que uma reta, o erro cai muito mais rápido.

Para que isso funcione, o número total de subintervalos deve ser **par**: agrupamos os subintervalos em $n$ pares, totalizando $2n$ subintervalos e $2n+1$ pontos. A fórmula do formulário é

$$\int_a^b f(x)\,dx \approx \frac{h}{3}\Big[f(x_0) + 4f(x_1) + 2f(x_2) + 4f(x_3) + \cdots + 2f(x_{2n-2}) + 4f(x_{2n-1}) + f(x_{2n})\Big]$$

onde os pontos de índice ímpar (vértices das parábolas) recebem peso 4 e os pontos de índice par internos (onde duas parábolas se encontram) recebem peso 2.

Como o erro local de cada par de subintervalos é $O(h^5)$ e há $n = O(1/h)$ pares, o erro total é $O(h^4)$ — **dobrar $n$ reduz o erro por um fator de aproximadamente 16**, muito mais rápido que os trapézios.

In [ ]:
def simpson(f, a, b, n_subintervalos):
    """Regra de Simpson repetida, implementada do zero.

    Observação de notação: aqui `n_subintervalos` é o número TOTAL de
    subintervalos (deve ser par), equivalente a 2n na notação do formulário
    (onde n é o número de pares de subintervalos / parábolas).

    Soma h/3 * [f(x0) + 4f(x1) + 2f(x2) + 4f(x3) + ... + f(x_{2n})]
    """
    if n_subintervalos % 2 != 0:
        raise ValueError('Simpson repetida exige um número PAR de subintervalos.')

    h = (b - a) / n_subintervalos
    x = a + h * np.arange(n_subintervalos + 1)
    y = f(x)

    # pesos: 1 nas pontas, 4 nos índices ímpares, 2 nos índices pares internos
    soma_impares = np.sum(y[1:-1:2])  # x1, x3, x5, ... (peso 4)
    soma_pares = np.sum(y[2:-1:2])    # x2, x4, x6, ... (peso 2, excluindo extremos)
    soma = y[0] + y[-1] + 4.0 * soma_impares + 2.0 * soma_pares
    return (h / 3.0) * soma


# Verificação rápida
n_check = 1000  # par
aprox_simpson = simpson(f_teste, a, b, n_check)
print(f'Simpson (implementação própria): {aprox_simpson:.10f}')
print(f'Valor exato                     : {exato:.10f}')

In [ ]:
def parabola_por_3_pontos(x0, y0, x1, y1, x2, y2, x_plot):
    """Retorna os valores y da parábola (polinômio de grau 2) que passa
    exatamente pelos 3 pontos dados, avaliada em x_plot. Usa interpolação
    de Lagrange — não depende de bibliotecas prontas de ajuste.
    """
    L0 = ((x_plot - x1) * (x_plot - x2)) / ((x0 - x1) * (x0 - x2))
    L1 = ((x_plot - x0) * (x_plot - x2)) / ((x1 - x0) * (x1 - x2))
    L2 = ((x_plot - x0) * (x_plot - x1)) / ((x2 - x0) * (x2 - x1))
    return y0 * L0 + y1 * L1 + y2 * L2


def plot_simpson(f, a, b, n_subintervalos, ax, titulo=''):
    """Desenha as parábolas interpoladoras usadas pela Regra de Simpson."""
    x_fino = np.linspace(a, b, 400)
    ax.plot(x_fino, f(x_fino), color=COR_FUNCAO, linewidth=2, label='$f(x)$')

    h = (b - a) / n_subintervalos
    x = a + h * np.arange(n_subintervalos + 1)
    y = f(x)

    primeira = True
    for i in range(0, n_subintervalos, 2):
        x0, x1, x2 = x[i], x[i + 1], x[i + 2]
        y0, y1, y2 = y[i], y[i + 1], y[i + 2]
        x_par = np.linspace(x0, x2, 50)
        y_par = parabola_por_3_pontos(x0, y0, x1, y1, x2, y2, x_par)
        ax.fill_between(x_par, 0, y_par, color=COR_SIMPSON, alpha=0.35,
                         label='Parábola de Simpson' if primeira else None)
        ax.plot(x_par, y_par, color=COR_SIMPSON, linewidth=1.5)
        primeira = False

    ax.scatter(x, y, color='k', zorder=5, s=20, label='Pontos usados')
    aproximacao = simpson(f, a, b, n_subintervalos)
    ax.set_title(f'{titulo}\n$2n={n_subintervalos}$ subintervalos, aproximação $\\approx {aproximacao:.5f}$')
    ax.set_xlabel('$x$')
    ax.set_ylabel('$f(x)$')
    ax.legend()


fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for ax, n_sub in zip(axs, [4, 8]):
    plot_simpson(f_teste, a, b, n_sub, ax, titulo='Regra de Simpson')

fig.suptitle('Parábolas interpoladoras aproximando $\\int_0^\\pi \\sin(x)\\,dx = 2$')
plt.tight_layout()
plt.show()

### Análise de convergência do erro

Repetimos o mesmo experimento de convergência, agora para a Regra de Simpson, esperando uma inclinação próxima de $-4$ no gráfico log-log (contra $-2$ dos trapézios).

In [ ]:
ns_pares = np.array([4, 8, 16, 32, 64, 128, 256, 512])  # número total de subintervalos (par)
erros_simpson = np.array([abs(simpson(f_teste, a, b, n) - exato) for n in ns_pares])

p_simpson, c_simpson = np.polyfit(np.log(ns_pares), np.log(erros_simpson), 1)
print(f'Ordem de convergência estimada (Simpson): p ~= {-p_simpson:.3f}  (teórico: 4)')

fig, ax = plt.subplots()
ax.loglog(ns_pares, erros_simpson, 's-', color=COR_SIMPSON, label='Erro numérico')

referencia_simpson = erros_simpson[0] * (ns_pares / ns_pares[0])**(-4.0)
ax.loglog(ns_pares, referencia_simpson, '--', color='gray', label='Referência $O(n^{-4})$')

ax.set_xlabel('$n$ (número de subintervalos)')
ax.set_ylabel('Erro absoluto')
ax.set_title(f'Convergência da Regra de Simpson (inclinação medida $\\approx {-p_simpson:.2f}$)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Comparação direta: Trapézios vs. Simpson

Vamos agora colocar os dois métodos no mesmo gráfico log-log, usando o mesmo problema-teste ($\int_0^\pi \sin(x)\,dx = 2$) e os mesmos valores de $n$ (número de subintervalos). A diferença de inclinação entre as duas curvas ($-2$ para Trapézios vs. $-4$ para Simpson) mostra visualmente por que a **ordem de convergência** de um método importa tanto na prática: para atingir uma precisão desejada, Simpson precisa de muito menos pontos (logo, muito menos avaliações de $f$) do que Trapézios.

In [ ]:
# Usamos a função e^x em [0, 1] como segundo problema-teste, com primitiva conhecida:
# integral exata de e^x em [0, 1] = e - 1
f_exp = np.exp
a2, b2 = 0.0, 1.0
exato2 = np.e - 1.0

ns_comp = np.array([4, 8, 16, 32, 64, 128, 256, 512])  # par, usado nos dois métodos

erros_trap_sin = np.array([abs(trapezios(f_teste, a, b, n) - exato) for n in ns_comp])
erros_simp_sin = np.array([abs(simpson(f_teste, a, b, n) - exato) for n in ns_comp])

erros_trap_exp = np.array([abs(trapezios(f_exp, a2, b2, n) - exato2) for n in ns_comp])
erros_simp_exp = np.array([abs(simpson(f_exp, a2, b2, n) - exato2) for n in ns_comp])

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))

axs[0].loglog(ns_comp, erros_trap_sin, 'o-', color=COR_TRAPEZIO, label='Trapézios ($O(n^{-2})$)')
axs[0].loglog(ns_comp, erros_simp_sin, 's-', color=COR_SIMPSON, label='Simpson ($O(n^{-4})$)')
axs[0].set_title('$\\int_0^\\pi \\sin(x)\\,dx = 2$')
axs[0].set_xlabel('$n$')
axs[0].set_ylabel('Erro absoluto')
axs[0].legend()

axs[1].loglog(ns_comp, erros_trap_exp, 'o-', color=COR_TRAPEZIO, label='Trapézios ($O(n^{-2})$)')
axs[1].loglog(ns_comp, erros_simp_exp, 's-', color=COR_SIMPSON, label='Simpson ($O(n^{-4})$)')
axs[1].set_title('$\\int_0^1 e^x\\,dx = e - 1$')
axs[1].set_xlabel('$n$')
axs[1].set_ylabel('Erro absoluto')
axs[1].legend()

fig.suptitle('Trapézios vs. Simpson: Simpson converge muito mais rápido (reta mais inclinada)')
plt.tight_layout()
plt.show()

## 7. Vá além: quando os métodos "sofrem" (malha uniforme não é sempre a melhor escolha)

Trapézios e Simpson, como implementados aqui, usam uma malha **uniforme** (pontos igualmente espaçados). Isso funciona muito bem quando $f$ é suave (poucas derivadas grandes). Mas existem funções em que a malha uniforme é ineficiente:

- Funções com **mudança brusca de curvatura** localizada (por exemplo, uma "corcova" estreita): a maior parte do intervalo é "fácil", mas uma pequena região exige muitos pontos para capturar a curvatura.
- Funções **oscilatórias rápidas**: se a malha não resolve bem as oscilações, o método pode convergir muito lentamente ou até dar resultados enganosos para $n$ pequeno.

Vamos ilustrar com uma função que tem um pico estreito e alto — uma gaussiana estreita $f(x) = e^{-100(x-0.5)^2}$ em $[0,1]$ — e comparar a convergência com a do caso suave ($e^x$) já estudado.

In [ ]:
def f_pico(x):
    """Gaussiana estreita: cresce e decai rapidamente perto de x=0.5."""
    return np.exp(-100.0 * (x - 0.5)**2)


a3, b3 = 0.0, 1.0
# Essa função não tem primitiva elementar (está ligada à função erro).
# Usamos Simpson com n muito grande como valor de referência "quase exato".
exato3 = simpson(f_pico, a3, b3, 200000)

fig, axs = plt.subplots(1, 2, figsize=(13, 5))

x_plot = np.linspace(a3, b3, 400)
axs[0].plot(x_plot, f_pico(x_plot), color=COR_FUNCAO, linewidth=2)
axs[0].set_title('$f(x) = e^{-100(x-0.5)^2}$: pico estreito')
axs[0].set_xlabel('$x$')
axs[0].set_ylabel('$f(x)$')

ns_pico = np.array([4, 8, 16, 32, 64, 128, 256, 512])
erros_trap_pico = np.array([abs(trapezios(f_pico, a3, b3, n) - exato3) for n in ns_pico])
erros_simp_pico = np.array([abs(simpson(f_pico, a3, b3, n) - exato3) for n in ns_pico])

axs[1].loglog(ns_pico, erros_trap_pico, 'o-', color=COR_TRAPEZIO, label='Trapézios')
axs[1].loglog(ns_pico, erros_simp_pico, 's-', color=COR_SIMPSON, label='Simpson')
axs[1].set_title('Convergência para a função de pico estreito')
axs[1].set_xlabel('$n$')
axs[1].set_ylabel('Erro absoluto (vs. referência)')
axs[1].legend()

plt.tight_layout()
plt.show()

Note que, para $n$ pequeno, a malha uniforme mal "enxerga" o pico (a largura efetiva do pico é da ordem de $0.1$, muito menor que o intervalo $[0,1]$), e o erro fica bem maior do que no caso suave do $e^x$ para o mesmo $n$ — as curvas de erro só se aproximam da inclinação teórica ($-2$ e $-4$) depois que $n$ é grande o suficiente para resolver a região de curvatura acentuada. Isso ilustra uma limitação prática importante: **métodos de malha uniforme "gastam" pontos igualmente em regiões fáceis e difíceis**. Na prática, métodos de **quadratura adaptativa** resolvem isso refinando a malha *localmente*, apenas onde a função varia mais rapidamente, concentrando o esforço computacional onde ele é realmente necessário — sem desperdiçar avaliações de $f$ em regiões já suaves. A implementação completa de um esquema adaptativo foge do escopo deste notebook, mas a ideia (refinar mais onde o erro estimado localmente é maior) é uma extensão natural do que vimos aqui.

## 8. Resumo

| Método | Forma geométrica | Fórmula | Ordem do erro |
|---|---|---|---|
| Soma de Riemann | Retângulos | $h\sum f(x_i)$ | $O(h)$ |
| Trapézios repetida | Trapézios (retas) | $\dfrac{h}{2}[f(x_0)+2f(x_1)+\cdots+2f(x_{n-1})+f(x_n)]$ | $O(h^2)$ |
| Simpson repetida | Parábolas | $\dfrac{h}{3}[f(x_0)+4f(x_1)+2f(x_2)+\cdots+4f(x_{2n-1})+f(x_{2n})]$ | $O(h^4)$ |

**Principais conclusões:**

- Quanto maior o grau do polinômio usado para aproximar localmente a função, maior a ordem de convergência do método — por isso Simpson (parábolas) vence Trapézios (retas) com folga.
- A ordem de convergência determina quantos pontos são necessários para atingir uma dada precisão: métodos de ordem mais alta exigem **muito** menos pontos (e, portanto, menos avaliações de $f$) para o mesmo erro.
- Malhas uniformes podem ser ineficientes para funções com comportamento muito localizado (picos estreitos, oscilações rápidas); nesses casos, refinar a malha de forma adaptativa é a estratégia mais eficiente.